# ✏️ → 🧊 Sketch2Shape AI — Colab runner

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then upload the project to `/content`
(either the `Sketch2Shape/` folder via Google Drive, or `Sketch2Shape.zip` via the Files sidebar).

**Then:** `Runtime → Run all`. Every cell is safe to re-run — finished work is skipped.
The last cell starts the web app and prints a public `https://….gradio.live` link.

| Cell | What it does | First run | Re-run |
|---|---|---|---|
| 1 | find project, check GPU | seconds | seconds |
| 2 | settings (VLM, quality) | — | — |
| 3 | install packages + models (+ Ollama/Gemma if chosen) | ~8–12 min | seconds |
| 4 | smoke test: chair sketch → 3D, shown inline | ~2–3 min | ~1 min |
| 5 | benchmark all 6 test sketches (optional) | ~5–8 min | ~5 min |
| 6 | save results (download + Drive) | seconds | seconds |
| 6b | **build final submission zip** (README filled with real timings) | seconds | seconds |
| 7 | **launch the web app** | ~1 min | ~1 min |

> Free Colab resets after ~90 min idle. Keep the tab active, and save results (cell 6) before you leave.

In [ ]:
# @title 1 · Find the project and check the GPU
import glob, os, shutil, subprocess, sys, zipfile
from pathlib import Path

USE_GOOGLE_DRIVE = False  # @param {type:"boolean"}
DRIVE_FOLDER = "/content/drive/MyDrive/Sketch2Shape"  # @param {type:"string"}

if USE_GOOGLE_DRIVE and not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

def _find_project():
    candidates = ["/content/Sketch2Shape", DRIVE_FOLDER] + glob.glob("/content/*/Sketch2Shape")
    for c in candidates:
        if c and os.path.exists(os.path.join(c, "app.py")):
            return Path(c)
    zips = sorted(glob.glob("/content/Sketch2Shape*.zip") + glob.glob("/content/drive/MyDrive/Sketch2Shape*.zip"),
                  key=os.path.getmtime)
    if zips:
        print("Extracting", zips[-1])
        zipfile.ZipFile(zips[-1]).extractall("/content")
        if os.path.exists("/content/Sketch2Shape/app.py"):
            return Path("/content/Sketch2Shape")
    return None

PROJECT = _find_project()
if PROJECT is None:
    raise SystemExit("Project not found. Upload the Sketch2Shape folder (or Sketch2Shape.zip) to /content, "
                     "or tick USE_GOOGLE_DRIVE and set DRIVE_FOLDER, then re-run this cell.")
# Work on local disk: Drive is slow for model caches and many small files.
if str(PROJECT).startswith("/content/drive"):
    print("Copying project from Drive to /content (faster)…")
    shutil.copytree(PROJECT, "/content/Sketch2Shape", dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("outputs", "third_party", "__pycache__"))
    PROJECT = Path("/content/Sketch2Shape")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print("Project:", PROJECT)

gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("No GPU. Runtime → Change runtime type → T4 GPU, then Run all again.")
ram = round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3, 1)
print(f"GPU: {gpu} | RAM: {ram} GB | Python {sys.version.split()[0]}")

In [ ]:
# @title 2 · Settings
import os
# @markdown **Sketch understanding model** – reads the drawing and names the object / parts / view.
VLM = "gemma3 4B (local, free)"  # @param ["gemma3 4B (local, free)", "claude (API key in Colab Secrets)", "clip (offline, fastest)"]
CLAUDE_MODEL = "claude-haiku-5-5"  # @param {type:"string"}
# @markdown **Image generation**
NUM_CANDIDATES = 2  # @param {type:"slider", min:1, max:4, step:1}
DIFFUSION_STEPS = 25  # @param {type:"slider", min:15, max:40, step:1}
CONTROLNET = "scribble"  # @param ["scribble", "lineart", "canny"]
# @markdown For coloured uploads: how much the image model may repaint your picture (lower = keeps your colours more)
IMG2IMG_STRENGTH = 0.75  # @param {type:"slider", min:0.5, max:0.95, step:0.05}
# @markdown **3D**
MESH_RESOLUTION = 256  # @param [192, 256, 320]
BAKE_UV_TEXTURE = False  # @param {type:"boolean"}
SMOOTHING_ITERATIONS = 8  # @param {type:"slider", min:0, max:20, step:1}
ENHANCE_COLOURS = True  # @param {type:"boolean"}

cfg = {
    "PYTHONUNBUFFERED": "1",
    "S2S_OFFLOAD": "auto",                    # keeps everything on a 15 GB T4; offloads on smaller GPUs
    "S2S_NUM_CANDIDATES": str(NUM_CANDIDATES),
    "S2S_STEPS": str(DIFFUSION_STEPS),
    "S2S_CONTROL_TYPE": CONTROLNET,
    "S2S_IMG2IMG_STRENGTH": str(IMG2IMG_STRENGTH),
    "S2S_MC_RES": str(MESH_RESOLUTION),
    "S2S_BAKE_TEXTURE": "1" if BAKE_UV_TEXTURE else "0",
    "S2S_SMOOTH_ITERS": str(SMOOTHING_ITERATIONS),
    "S2S_ENHANCE_COLORS": "1" if ENHANCE_COLOURS else "0",
    "S2S_VLM_TIMEOUT": "180",                 # first Gemma call loads it from disk
}
if VLM.startswith("gemma"):
    cfg.update(S2S_VLM_BACKEND="openai", OPENAI_BASE_URL="http://localhost:11434/v1",
               OPENAI_MODEL="gemma3:4b", OPENAI_API_KEY="ollama")
elif VLM.startswith("claude"):
    from google.colab import userdata
    try:
        cfg.update(S2S_VLM_BACKEND="anthropic", ANTHROPIC_API_KEY=userdata.get("ANTHROPIC_API_KEY"),
                   ANTHROPIC_MODEL=CLAUDE_MODEL)
    except Exception as exc:
        raise SystemExit("Add ANTHROPIC_API_KEY in Colab Secrets (🔑 sidebar) with Notebook access ON. "
                         f"({exc.__class__.__name__})")
else:
    cfg.update(S2S_VLM_BACKEND="clip")
os.environ.update(cfg)
print("Settings applied:", {k: ("***" if "KEY" in k else v) for k, v in cfg.items()})

In [ ]:
# @title 3 · Install (skips anything already installed)
import glob, json, os, shutil, subprocess, sys, time
from pathlib import Path
from IPython.display import HTML, Image as IImage, Markdown, display
PROJECT = Path("/content/Sketch2Shape")
if not (PROJECT / "app.py").exists():
    raise SystemExit("Run cell 1 first (project not found in /content/Sketch2Shape).")
os.chdir(PROJECT); sys.path.insert(0, str(PROJECT))
import importlib.util, requests

def _pkgs_ok():
    return all(importlib.util.find_spec(m) for m in ("diffusers", "gradio", "rembg", "trimesh", "mcubes"))

def _models_ok():
    hub = Path.home() / ".cache/huggingface/hub"
    return all((hub / f"models--{r}").exists() for r in
               ("Lykon--dreamshaper-8", f"lllyasviel--control_v11p_sd15_{os.environ['S2S_CONTROL_TYPE']}",
                "stabilityai--TripoSR"))

t0 = time.time()
if _pkgs_ok() and (PROJECT / "third_party/TripoSR/tsr").exists() and _models_ok():
    print("✓ Packages, TripoSR and model weights already installed")
else:
    !bash scripts/setup.sh 2>&1 | grep --line-buffered -v "^ERROR: pip's dependency resolver\|is incompatible\.$"
    if not _pkgs_ok():
        raise SystemExit("Setup did not finish - scroll up for the first real error.")

# ---- local VLM via Ollama (only if Gemma was chosen) --------------------------------------------
def _ollama_up():
    try:
        return requests.get("http://localhost:11434", timeout=2).ok
    except Exception:
        return False

if os.environ.get("OPENAI_MODEL") == "gemma3:4b":
    if not shutil.which("ollama"):
        !apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils > /dev/null
        !curl -fsSL https://ollama.com/install.sh | sh 2>&1 | grep --line-buffered -v "systemd"
    if not _ollama_up():
        subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
        for _ in range(60):
            if _ollama_up():
                break
            time.sleep(1)
    if not _ollama_up():
        raise SystemExit("Ollama did not start: !tail -30 /content/ollama.log")
    if "gemma3:4b" not in subprocess.run("ollama list", shell=True, capture_output=True, text=True).stdout:
        !ollama pull gemma3:4b
    print("✓ Ollama running with gemma3:4b",
          "(GPU)" if "CUDA" in open("/content/ollama.log").read() else "(check GPU in /content/ollama.log)")

print(f"✓ Ready in {time.time() - t0:.0f}s")

In [ ]:
# @title 4 · Smoke test: chair sketch → 3D model (shown inline)
SAMPLE = "samples/02_chair.png"  # @param ["samples/01_cup.png", "samples/02_chair.png", "samples/03_table_lamp.png", "samples/04_car_rough.png", "samples/05_noisy_photo_vase.png", "samples/06_house_chalkboard.png"]
PROMPT = "a wooden chair"  # @param {type:"string"}
import glob, json, os, shutil, subprocess, sys, time
from pathlib import Path
from IPython.display import HTML, Image as IImage, Markdown, display
PROJECT = Path("/content/Sketch2Shape")
if not (PROJECT / "app.py").exists():
    raise SystemExit("Run cell 1 first (project not found in /content/Sketch2Shape).")
os.chdir(PROJECT); sys.path.insert(0, str(PROJECT))

!python scripts/run_cli.py "$SAMPLE" --prompt "$PROMPT" --candidates 1 2>&1 | grep --line-buffered -v "Flax classes\|CLIPFeatureExtractor\|absl::\|cudart_stub"

reports = sorted(glob.glob("outputs/jobs/*/report.json"), key=os.path.getmtime)
if not reports:
    raise SystemExit("No job finished - read the log above.")
job_dir = Path(reports[-1]).parent
rep = json.load(open(reports[-1]))
if rep.get("error") or "glb" not in rep["files"]:
    raise SystemExit(f"Run failed: {rep.get('error')}")

print("\nObject:", rep["sketch_understanding"]["object"], "| via", rep["sketch_understanding"]["source"])
print("Timings (s):", rep["timings_s"])
m = rep["mesh"]["final"]
print(f"Mesh: {m['num_vertices']:,} vertices, {m['num_faces']:,} faces, {m['visual_kind']} colour, "
      f"GLB re-load {'OK' if rep['mesh']['export_check']['readable'] else 'FAILED'}")
best = next(c for c in rep["candidates"] if c["index"] == rep["selected_candidate"])
display(HTML("<b>input → ControlNet map → generated image</b>"))
for f in (rep["files"]["original"], rep["files"]["control"], best["path"]):
    display(IImage(f, width=230))
from frontend.viewer import build_viewer_html, iframe
display(HTML(iframe(build_viewer_html(Path(rep["files"]["glb"])), height=460)))

In [ ]:
# @title 5 · Benchmark all 6 test sketches (real timings for the README)
RUN_BENCHMARK = True  # @param {type:"boolean"}
import glob, json, os, shutil, subprocess, sys, time
from pathlib import Path
from IPython.display import HTML, Image as IImage, Markdown, display
PROJECT = Path("/content/Sketch2Shape")
if not (PROJECT / "app.py").exists():
    raise SystemExit("Run cell 1 first (project not found in /content/Sketch2Shape).")
os.chdir(PROJECT); sys.path.insert(0, str(PROJECT))
if RUN_BENCHMARK:
    !python scripts/benchmark.py --warmup 2>&1 | grep --line-buffered -E "^###|FAILED|Saved|^\|"
    display(Markdown(open("outputs/benchmark/results.md").read()))
    for d in sorted(glob.glob("outputs/benchmark/*/")):
        display(HTML(f"<b>{Path(d).name}</b>"))
        display(*[IImage(d + f, width=180) for f in ("input.png", "control_map.png", "generated.png")
                  if os.path.exists(d + f)])
else:
    print("Skipped (tick RUN_BENCHMARK to run).")

In [ ]:
# @title 6 · Save results (download + optional Google Drive copy)
COPY_TO_DRIVE = False  # @param {type:"boolean"}
import glob, json, os, shutil, subprocess, sys, time
from pathlib import Path
from IPython.display import HTML, Image as IImage, Markdown, display
PROJECT = Path("/content/Sketch2Shape")
if not (PROJECT / "app.py").exists():
    raise SystemExit("Run cell 1 first (project not found in /content/Sketch2Shape).")
os.chdir(PROJECT); sys.path.insert(0, str(PROJECT))
out = Path("/content/sketch2shape_results")
shutil.rmtree(out, ignore_errors=True)
out.mkdir()
if os.path.exists("outputs/benchmark"):
    shutil.copytree("outputs/benchmark", out / "benchmark")
for rp in sorted(glob.glob("outputs/jobs/*/report.json"), key=os.path.getmtime)[-10:]:   # last 10 jobs
    jd = Path(rp).parent
    dst = out / "jobs" / jd.name
    dst.mkdir(parents=True)
    for f in ["report.json", "00_original.png", "07_control_map.png", "13_recon_input.png"] + \
             [Path(p).name for p in glob.glob(str(jd / "08_candidate_*.png"))]:
        if (jd / f).exists():
            shutil.copy(jd / f, dst / f)
    if (jd / "export").exists():
        shutil.copytree(jd / "export", dst / "export")
archive = shutil.make_archive("/content/sketch2shape_results", "zip", out)
print("Saved", archive, f"({os.path.getsize(archive) / 1024**2:.1f} MB)")
if COPY_TO_DRIVE:
    if not os.path.exists("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/content/drive")
    shutil.copy(archive, "/content/drive/MyDrive/")
    print("Copied to Google Drive / MyDrive")
from google.colab import files
files.download(archive)

In [ ]:
# @title 6b · Build the final submission zip (after cell 5)
# @markdown Copies the benchmark GLBs + images into `outputs/submission/`, writes the measured results into
# @markdown README.md, re-checks every GLB, and zips code + docs + artefacts (no model weights).
SAVE_TO_DRIVE = True  # @param {type:"boolean"}
import glob, json, os, shutil, subprocess, sys, time
from pathlib import Path
PROJECT = Path("/content/Sketch2Shape")
if not (PROJECT / "app.py").exists():
    raise SystemExit("Run cell 1 first (project not found in /content/Sketch2Shape).")
os.chdir(PROJECT); sys.path.insert(0, str(PROJECT))
!python scripts/finalize_submission.py
zip_path = "/content/Sketch2Shape_submission.zip"
if SAVE_TO_DRIVE:
    if not os.path.exists("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/content/drive")
    shutil.copy(zip_path, "/content/drive/MyDrive/")
    print("Copied to Google Drive / MyDrive / Sketch2Shape_submission.zip")
from google.colab import files
files.download(zip_path)


In [ ]:
# @title 7 · Launch the web app (public link) — stop with ■ when done
# @markdown Wait for **Running on public URL** (~1 min), open it, draw, press *Generate 3D model*.
# @markdown Share the link only with your team / judges: anyone with it uses this GPU.
import glob, json, os, shutil, subprocess, sys, time
from pathlib import Path
from IPython.display import HTML, Image as IImage, Markdown, display
PROJECT = Path("/content/Sketch2Shape")
if not (PROJECT / "app.py").exists():
    raise SystemExit("Run cell 1 first (project not found in /content/Sketch2Shape).")
os.chdir(PROJECT); sys.path.insert(0, str(PROJECT))
!python app.py --share --preload 2>&1 | grep --line-buffered -v "Flax classes\|CLIPFeatureExtractor\|absl::\|cudart_stub"

### Troubleshooting
* **Link shows "Broken connection"** → look at the end of cell 7's log. Run the diagnostics cell below and send its output.
* **`ollama: command not found` / Gemma errors** → the runtime was reset: Run all again (installs are skipped where possible).
* **Wrong object recognised** → type the object name in *Describe the object*; your words override the VLM.
* **Out of memory** → in cell 2 set candidates to 1 and mesh resolution to 192, re-run cell 2, then cell 7.

In [ ]:
# @title Diagnostics (run when something goes wrong)
import glob, json, os, shutil, subprocess, sys, time
from pathlib import Path
from IPython.display import HTML, Image as IImage, Markdown, display
PROJECT = Path("/content/Sketch2Shape")
if not (PROJECT / "app.py").exists():
    raise SystemExit("Run cell 1 first (project not found in /content/Sketch2Shape).")
os.chdir(PROJECT); sys.path.insert(0, str(PROJECT))
!free -h | head -2
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
!command -v ollama > /dev/null && ollama ps
!tail -15 /content/ollama.log 2>/dev/null
!ls -t outputs/jobs 2>/dev/null | head -3